# RideWise -- Data Preprocessing

Fixing all data quality issues identified during our EDA prepare clean datset fot feature engineering & Modelling

In [25]:
!pip install matplotlib seaborn

In [26]:
## Setup & import libraries

import os, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


warnings.filterwarnings("ignore")
%matplotlib inline
sns.set_theme(style="whitegrid", font_scale=1.2)


## define the location of our raw and processed 
DATA_RAW = os.path.join("..", "data", "raw")
DATA_PROCESSED = os.path.join("..", "data", "processed")
os.makedirs(DATA_PROCESSED, exist_ok=True)

###### RIDEWISE_LONDON-MUHAMMAD/
- data/
   - raw/
   - processed/

In [27]:
# Loading the datasets
riders = pd.read_csv(os.path.join(DATA_RAW, "riders.csv"), parse_dates=["signup_date"])
trips = pd.read_csv(os.path.join(DATA_RAW, "trips.csv"))
drivers = pd.read_csv(os.path.join(DATA_RAW, "drivers.csv"), parse_dates=["signup_date"])
sessions = pd.read_csv(os.path.join(DATA_RAW, "sessions.csv"))
promotions = pd.read_csv(os.path.join(DATA_RAW, "promotions.csv"))


### Step 1 Cleaning Exercise (Riders Dataset)

In [28]:
riders.head()

,user_id,signup_date,loyalty_status,age,city,avg_rating_given,churn_prob,referred_by
0,R00000,2025-01-24,Bronze,34.729629,Nairobi,5.0,0.142431,R00001
1,R00001,2024-09-09,Bronze,34.571020,Nairobi,4.7,0.674161,NaN
2,R00002,2024-09-07,Bronze,47.133960,Lagos,4.2,0.510379,NaN
3,R00003,2025-03-17,Bronze,41.658628,Nairobi,4.9,0.244779,NaN
4,R00004,2024-08-20,Silver,40.681709,Lagos,3.9,0.269960,R00002


- 1 - Reffered by 
- 2 - signup date
- 3 - churn prob (threshold)
- 4 - age  --- (ranges)

In [29]:
riders.isnull().sum()

user_id                0
signup_date            0
loyalty_status         0
age                    0
city                   0
avg_rating_given       0
churn_prob             0
referred_by         6947
dtype: int64

In [30]:
# Make a copy of the original dataframes to work with
riders_clean = riders.copy()

# STEP 1A -- fix age (round up the values)
riders_clean["age"] = riders_clean["age"].round().astype(int)


# STEP 1B -- fix churn label (converting from continuous prob to binary label) : e.g (0.124, 0.4358 etc) OR (1 & 0)
# STEP 1B(I) -- create a threshold (0.5 ~ 50%) : churn probablity is > 50% = churned OR churn probability is > 50% = Not churned 
riders_clean["churned"] = (riders_clean["churn_prob"] > 0.5).astype(int)
riders_clean = riders_clean.drop(columns=["churn_prob"])


# STEP 1C- fix the reffered by option fill with "Unknown" or "Not reffered"
# option 2 return the ones that were referred (binaty signal instead of racking my head on who was referred or not)
riders_clean["was_referred"] = riders_clean["referred_by"].notna().astype(int)
riders_clean = riders_clean.drop(columns=["referred_by"])

# check for formatting
print(f"Riders columns data-types :{riders_clean.dtypes}")
riders_clean.head()

Riders columns data-types :user_id                     object
signup_date         datetime64[ns]
loyalty_status              object
age                          int32
city                        object
avg_rating_given           float64
churned                      int32
was_referred                 int32
dtype: object


,user_id,signup_date,loyalty_status,age,city,avg_rating_given,churned,was_referred
0,R00000,2025-01-24,Bronze,35,Nairobi,5.0,0,1
1,R00001,2024-09-09,Bronze,35,Nairobi,4.7,1,0
2,R00002,2024-09-07,Bronze,47,Lagos,4.2,1,0
3,R00003,2025-03-17,Bronze,42,Nairobi,4.9,0,0
4,R00004,2024-08-20,Silver,41,Lagos,3.9,0,1


### Step 2 - Cleaning Exercise (Trips)

In [31]:
trips.head()

,trip_id,user_id,driver_id,fare,surge_multiplier,tip,payment_type,pickup_time,dropoff_time,pickup_lat,pickup_lng,dropoff_lat,dropoff_lng,weather,city,loyalty_status
0,T000000,R05207,D00315,12.11,1.0,0.00,Card,2024-11-27 18:41:50+02:27,2024-11-27 19:33:50+02:27,-1.108123,36.912209,-1.068155,36.875377,Foggy,Nairobi,Bronze
1,T000001,R09453,D03717,8.73,1.0,0.02,Card,2024-10-28 23:13:48+00:14,2024-10-28 23:26:48+00:14,6.675266,3.515740,6.641734,3.525620,Sunny,Lagos,Gold
2,T000002,R00567,D02035,19.68,1.0,0.00,Card,2025-02-17 05:36:41+02:27,2025-02-17 05:52:41+02:27,-1.248589,37.010668,-1.273182,37.018586,Cloudy,Nairobi,Bronze
3,T000003,R09573,D02657,16.43,1.0,0.01,Mobile Money,2024-06-18 19:27:14+02:05,2024-06-18 19:32:14+02:05,29.819554,31.188780,29.837689,31.232978,Cloudy,Cairo,Bronze
4,T000004,R03446,D01026,8.70,1.0,1.06,Card,2024-10-05 09:58:16+02:27,2024-10-05 10:28:16+02:27,-1.676479,36.729219,-1.638395,36.694063,Sunny,Nairobi,Gold


In [32]:
# copy the trips dataset
trips_clean = trips.copy()

# STEP 2A -- fix the date columns (convert to datetime) (drop_off and pick_up)
for col in ["pickup_time", "dropoff_time"]:
    trips_clean[col] = (
        pd.to_datetime(trips_clean[col], utc=True, errors="coerce").dt.tz_localize(None)
    )
trips_clean = trips_clean.dropna(subset=["pickup_time", "dropoff_time"])


# print out all attributes
print(f"Trips columns data-types :{trips_clean.dtypes}")
trips_clean.head()



Trips columns data-types :trip_id                     object
user_id                     object
driver_id                   object
fare                       float64
surge_multiplier           float64
tip                        float64
payment_type                object
pickup_time         datetime64[ns]
dropoff_time        datetime64[ns]
pickup_lat                 float64
pickup_lng                 float64
dropoff_lat                float64
dropoff_lng                float64
weather                     object
city                        object
loyalty_status              object
dtype: object


,trip_id,user_id,driver_id,fare,surge_multiplier,tip,payment_type,pickup_time,dropoff_time,pickup_lat,pickup_lng,dropoff_lat,dropoff_lng,weather,city,loyalty_status
0,T000000,R05207,D00315,12.11,1.0,0.00,Card,2024-11-27 16:14:50,2024-11-27 17:06:50,-1.108123,36.912209,-1.068155,36.875377,Foggy,Nairobi,Bronze
1,T000001,R09453,D03717,8.73,1.0,0.02,Card,2024-10-28 22:59:48,2024-10-28 23:12:48,6.675266,3.515740,6.641734,3.525620,Sunny,Lagos,Gold
2,T000002,R00567,D02035,19.68,1.0,0.00,Card,2025-02-17 03:09:41,2025-02-17 03:25:41,-1.248589,37.010668,-1.273182,37.018586,Cloudy,Nairobi,Bronze
3,T000003,R09573,D02657,16.43,1.0,0.01,Mobile Money,2024-06-18 17:22:14,2024-06-18 17:27:14,29.819554,31.188780,29.837689,31.232978,Cloudy,Cairo,Bronze
4,T000004,R03446,D01026,8.70,1.0,1.06,Card,2024-10-05 07:31:16,2024-10-05 08:01:16,-1.676479,36.729219,-1.638395,36.694063,Sunny,Nairobi,Gold


In [33]:
# Engineered some new features from the trips dataset

# 1. Trip duration (in minutes)
trips_clean["trip_duration"] = (trips_clean["dropoff_time"] - trips_clean["pickup_time"]).dt.total_seconds() / 60

# 2. Total revenue (fare * surge_multiplier + tip)
trips_clean["total_revenue"] = (
    trips_clean["fare"] * trips_clean["surge_multiplier"] + trips_clean["tip"].fillna(0)
)

# 3 . Hour of the day
trips_clean["hour_of_day"] = trips_clean["pickup_time"].dt.hour

# 4. Day of the week
trips_clean["day_of_week"] = trips_clean["pickup_time"].dt.day_name()


In [34]:
# Data quality in-check
trips_clean = trips_clean[trips_clean["trip_duration"] > 0]  # Remove trips with non-positive duration
trips_clean["tip"] = trips_clean["tip"].fillna(0)  # Fill missing tips with 0
trips_clean["weather"] = trips_clean["weather"].fillna("unknown")


print(f"Rows shape: {trips_clean.shape}")
trips_clean.head()

Rows shape: (200000, 20)


,trip_id,user_id,driver_id,fare,surge_multiplier,tip,payment_type,pickup_time,dropoff_time,pickup_lat,pickup_lng,dropoff_lat,dropoff_lng,weather,city,loyalty_status,trip_duration,total_revenue,hour_of_day,day_of_week
0,T000000,R05207,D00315,12.11,1.0,0.00,Card,2024-11-27 16:14:50,2024-11-27 17:06:50,-1.108123,36.912209,-1.068155,36.875377,Foggy,Nairobi,Bronze,52.0,12.11,16,Wednesday
1,T000001,R09453,D03717,8.73,1.0,0.02,Card,2024-10-28 22:59:48,2024-10-28 23:12:48,6.675266,3.515740,6.641734,3.525620,Sunny,Lagos,Gold,13.0,8.75,22,Monday
2,T000002,R00567,D02035,19.68,1.0,0.00,Card,2025-02-17 03:09:41,2025-02-17 03:25:41,-1.248589,37.010668,-1.273182,37.018586,Cloudy,Nairobi,Bronze,16.0,19.68,3,Monday
3,T000003,R09573,D02657,16.43,1.0,0.01,Mobile Money,2024-06-18 17:22:14,2024-06-18 17:27:14,29.819554,31.188780,29.837689,31.232978,Cloudy,Cairo,Bronze,5.0,16.44,17,Tuesday
4,T000004,R03446,D01026,8.70,1.0,1.06,Card,2024-10-05 07:31:16,2024-10-05 08:01:16,-1.676479,36.729219,-1.638395,36.694063,Sunny,Nairobi,Gold,30.0,9.76,7,Saturday


### Step 3 - Cleaning Exercise for Drivers (Dataset)

In [35]:
drivers_clean = drivers.copy()
drivers_clean.head()

,driver_id,rating,vehicle_type,signup_date,last_active,city,acceptance_rate
0,D00000,3.1,SUV,2025-01-20,2025-01-06 18:23:09.312275,Cairo,0.679555
1,D00001,5.0,Sedan,2023-03-27,2025-04-27 01:44:02.472554,Nairobi,0.548786
2,D00002,4.5,Motorcycle,2024-05-02,2025-03-07 19:24:46.367672,Nairobi,0.593724
3,D00003,5.0,Motorcycle,2023-04-16,2025-03-26 19:16:24.253793,Nairobi,0.990000
4,D00004,4.4,Motorcycle,2023-05-28,2025-04-08 18:54:44.649615,Lagos,0.519773


In [36]:
drivers_clean['rating'] = drivers_clean['rating'].fillna(drivers_clean['rating'].median())
drivers_clean['acceptance_rate'] = drivers_clean['acceptance_rate'].fillna(drivers_clean['acceptance_rate'].median())

print(drivers_clean.dtypes)
print(drivers_clean.isnull().sum())

driver_id                  object
rating                    float64
vehicle_type               object
signup_date        datetime64[ns]
last_active                object
city                       object
acceptance_rate           float64
dtype: object
driver_id          0
rating             0
vehicle_type       0
signup_date        0
last_active        0
city               0
acceptance_rate    0
dtype: int64


### Step 4 - Cleaning Exercise : Session Dataset


In [37]:
# step 4(1) -- fix the session_time date format
sessions_clean = sessions.copy()
sessions_clean["session_time"] = (
    pd.to_datetime(sessions_clean["session_time"], utc=True, errors="coerce").dt.tz_localize(None)
)

# step 4(2) -- drop rows with invalid session_time
sessions_clean = sessions_clean.dropna(subset=["session_time"])

print(f"sessions_clean  : {sessions_clean.shape}")
sessions_clean.head()

sessions_clean  : (50000, 8)


,session_id,rider_id,session_time,time_on_app,pages_visited,converted,city,loyalty_status
0,S000000,R08605,2025-04-27 16:52:06,79,4,1,Cairo,Bronze
1,S000001,R08823,2025-04-27 05:05:22,101,3,0,Nairobi,Silver
2,S000002,R05342,2025-04-27 21:12:25,12,1,0,Cairo,Bronze
3,S000003,R05057,2025-04-27 14:26:25,19,1,0,Lagos,Silver
4,S000004,R09614,2025-04-27 08:17:22,4,1,0,Lagos,Bronze


### Step 5 - Referential Integrity 

In [38]:
# validating the  riders and drivers datasets
valid_riders = set(riders_clean["user_id"])
valid_drivers = set(drivers_clean["driver_id"])


# check the trips & session dataset
before_trips = len(trips_clean)
before_sessions = len(sessions_clean)


# filter to check if riders are in trips dataset
trips_clean = trips_clean[trips_clean["user_id"].isin(valid_riders)]
trips_clean = trips_clean[trips_clean["driver_id"].isin(valid_drivers)]
sessions_clean = sessions_clean[sessions_clean["rider_id"].isin(valid_riders)]

# check the descripancies
print(f"Trips that were dropped : {before_trips - len(trips_clean)}")
print(f"Sessions that were dropped : {before_sessions - len(sessions_clean)}")
print("Everything looks good")



Trips that were dropped : 0
Sessions that were dropped : 0
Everything looks good


##  Validate & Save

In [39]:
print("Row count - raw vs clean")
for name, raw, clean in [
    ("riders", riders, riders_clean),
    ("drivers", drivers, drivers_clean),
    ("trips", trips, trips_clean),
    ("sessions", sessions, sessions_clean)
]:
    print(f"  {name:<10}: {len(raw):>7,} -> {len(clean):>7,}")

Row count - raw vs clean
  riders    :  10,000 ->  10,000
  drivers   :   5,000 ->   5,000
  trips     : 200,000 -> 200,000
  sessions  :  50,000 ->  50,000


In [40]:
trips_clean.head(5)

,trip_id,user_id,driver_id,fare,surge_multiplier,tip,payment_type,pickup_time,dropoff_time,pickup_lat,pickup_lng,dropoff_lat,dropoff_lng,weather,city,loyalty_status,trip_duration,total_revenue,hour_of_day,day_of_week
0,T000000,R05207,D00315,12.11,1.0,0.00,Card,2024-11-27 16:14:50,2024-11-27 17:06:50,-1.108123,36.912209,-1.068155,36.875377,Foggy,Nairobi,Bronze,52.0,12.11,16,Wednesday
1,T000001,R09453,D03717,8.73,1.0,0.02,Card,2024-10-28 22:59:48,2024-10-28 23:12:48,6.675266,3.515740,6.641734,3.525620,Sunny,Lagos,Gold,13.0,8.75,22,Monday
2,T000002,R00567,D02035,19.68,1.0,0.00,Card,2025-02-17 03:09:41,2025-02-17 03:25:41,-1.248589,37.010668,-1.273182,37.018586,Cloudy,Nairobi,Bronze,16.0,19.68,3,Monday
3,T000003,R09573,D02657,16.43,1.0,0.01,Mobile Money,2024-06-18 17:22:14,2024-06-18 17:27:14,29.819554,31.188780,29.837689,31.232978,Cloudy,Cairo,Bronze,5.0,16.44,17,Tuesday
4,T000004,R03446,D01026,8.70,1.0,1.06,Card,2024-10-05 07:31:16,2024-10-05 08:01:16,-1.676479,36.729219,-1.638395,36.694063,Sunny,Nairobi,Gold,30.0,9.76,7,Saturday


In [41]:
files = {
    "riders_clean.csv": riders_clean,
    "trips_clean.csv": trips_clean,
    "drivers_clean.csv": drivers_clean,
    "sessions_clean.csv": sessions_clean
}

for fname, df in files.items():
    df.to_csv(os.path.join(DATA_PROCESSED, fname), index=False)
    print("Data saved successfully")

Data saved successfully
Data saved successfully
Data saved successfully
Data saved successfully
